In [ ]:
!pip install rasterio

In [ ]:
import rasterio
import numpy as np

Loading the bands

It loads the raw satellite bands b02,b03,b04,b08 which are used to create dataset for trainig the model.

In [ ]:
import rasterio

b02_path = "/content/B02.tiff"
b03_path = "/content/B03.tiff"
b04_path = "/content/B04.tiff"
b08_path = "/content/B08.tiff"

with rasterio.open(b02_path) as src:
    print("B02:", src.count, src.crs)

with rasterio.open(b03_path) as src:
    print("B03:", src.count, src.crs)

with rasterio.open(b04_path) as src:
    print("B04:", src.count, src.crs)

with rasterio.open(b08_path) as src:
    print("B08:", src.count, src.crs)

B02: 1 EPSG:4326
B03: 1 EPSG:4326
B04: 1 EPSG:4326
B08: 1 EPSG:4326


In [ ]:
with rasterio.open(b02_path) as src:
    B02 = src.read(1)

with rasterio.open(b03_path) as src:
    B03 = src.read(1)

with rasterio.open(b04_path) as src:
    B04 = src.read(1)

with rasterio.open(b08_path) as src:
    B08 = src.read(1)

print("B03 shape:", B03.shape)
print("B04 shape:", B04.shape)
print("B08 shape:", B08.shape)

B03 shape: (1188, 2500)
B04 shape: (1188, 2500)
B08 shape: (1188, 2500)


Computing NDWI and NDTI

Using the raw satellite bands NDWI and NDTI is calculated ?

In [ ]:
ndwi = np.full(B03.shape, np.nan, dtype="float32")
ndti = np.full(B04.shape, np.nan, dtype="float32")

valid_1 = (B03 + B04) != 0
valid = (B03 + B08) != 0

ndwi[valid] = (B03[valid] - B08[valid]) / (B03[valid] + B08[valid])
ndti[valid_1] = (B04[valid_1] - B03[valid_1]) / (B04[valid_1] + B03[valid_1])


print("NDWI shape:", ndwi.shape)
print("NDWI min:", np.nanmin(ndwi))
print("NDWI max:", np.nanmax(ndwi))

print("\nNDTI shape:", ndti.shape)
print("NDTI min:", np.nanmin(ndti))
print("NDTI max:", np.nanmax(ndti))

NDWI shape: (1188, 2500)
NDWI min: -1.0
NDWI max: 1.0

NDTI shape: (1188, 2500)
NDTI min: -1.0
NDTI max: 1.0


In [ ]:
stacked_data = np.stack([B02, B03, B04, B08, ndwi, ndti], axis=0)

print("Stacked shape:", stacked_data.shape)

Stacked shape: (6, 1188, 2500)


Cutting 32×32 patches

we have taken only 32x32 patches to train the model on some data for faster results ?

In [ ]:
patch = stacked_data[:, :32, :32]

print('Patch shape:', patch.shape)

Patch shape: (6, 32, 32)


In [ ]:
patches = []
coords = []


for row in range(0, 1188 - 32 + 1, 32):
    for col in range(0, 2500 - 32 + 1, 32):

        patch = stacked_data[:, row:row+32, col:col+32]

        patches.append(patch)
        coords.append((row, col))

print("Total patches:", len(patches))

print("Patch 1 shape:", patches[0].shape)
print("Patch 2 shape:", patches[1].shape)
print("Patch 3 shape:", patches[2].shape)

Total patches: 2886
Patch 1 shape: (6, 32, 32)
Patch 2 shape: (6, 32, 32)
Patch 3 shape: (6, 32, 32)


In [ ]:
import torch
from torch.utils.data import Dataset

In [ ]:
from rasterio.transform import xy

def pixel_to_latlon(transform, row, col):
    lon, lat = xy(transform, row, col, offset="center")
    return lat, lon

Dataset class and NTU labels

?

In [ ]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    dlat = lat2 - lat1
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c


def nearest_sensor(lat, lon, sensors):
    best_sensor = None
    best_distance = float("inf")

    for sensor in sensors:
        distance = haversine_km(
            lat,
            lon,
            sensor["lat"],
            sensor["lon"]
        )

        if distance < best_distance:
            best_distance = distance
            best_sensor = sensor

    return best_sensor

In [ ]:
class FusionDataset(Dataset):

    def __init__(self, patches, coords, transform, sensors):
        self.patches = patches
        self.coords = coords
        self.transform = transform
        self.sensors = sensors

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, index):

        patch = self.patches[index]

        row, col = self.coords[index]

        center_row = row + 16
        center_col = col + 16

        lat, lon = pixel_to_latlon(
            self.transform,
            center_row,
            center_col
        )

        sensor = nearest_sensor(
            lat,
            lon,
            self.sensors
        )

        ntu = sensor["ntu"]

        patch = torch.tensor(
            patch,
            dtype=torch.float32
        )

        ntu = torch.tensor(
            ntu,
            dtype=torch.float32
        )

        return patch, ntu

In [ ]:
import rasterio

with rasterio.open("/content/Jul_polluted.tif") as src:
    transform = src.transform

In [ ]:
import csv

sensors = []

with open("/content/Jul_sensors.csv", newline="", encoding="utf-8") as f:
    for rec in csv.DictReader(f):
        sensors.append({
            "station_id": rec["Station_ID"],
            "lat": float(rec["Lat"]),
            "lon": float(rec["Lon"]),
            "ntu": float(rec["Turbidity_NTU"])
        })

Dataset is created

?

In [ ]:
dataset = FusionDataset(
    patches,
    coords,
    transform,
    sensors
)

print("Number of samples:", len(dataset))

patch, ntu = dataset[0]

print("Patch type:", type(patch))
print("Patch shape:", patch.shape)

print("NTU type:", type(ntu))
print("NTU value:", ntu)

Number of samples: 2886
Patch type: <class 'torch.Tensor'>
Patch shape: torch.Size([6, 32, 32])
NTU type: <class 'torch.Tensor'>
NTU value: tensor(5.)


CNN architecture

?

In [ ]:
import torch
import torch.nn as nn


class NTUCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=6,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.fc = nn.Linear(64 * 8 * 8, 1)

    def forward(self, x):

        x = torch.relu(self.conv1(x))
        x = self.pool(x)

        x = torch.relu(self.conv2(x))
        x = self.pool(x)

        x = torch.flatten(x, start_dim=1)

        x = self.fc(x)

        return x

In [ ]:
model = NTUCNN()

print(model)

NTUCNN(
  (conv1): Conv2d(6, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=4096, out_features=1, bias=True)
)


In [ ]:
patch, ntu = dataset[0]

output = model(patch.unsqueeze(0))

print("Input shape:", patch.shape)
print("Output shape:", output.shape)
print("Predicted NTU:", output)

Input shape: torch.Size([6, 32, 32])
Output shape: torch.Size([1, 1])
Predicted NTU: tensor([[-0.0349]], grad_fn=<AddmmBackward0>)


In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

In [ ]:
images, labels = next(iter(loader))
labels = labels.unsqueeze(1)

outputs = model(images)

print("Input shape:", images.shape)
print("Label shape:", labels.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([8, 6, 32, 32])
Label shape: torch.Size([8, 1])
Output shape: torch.Size([8, 1])


Loss function, optimizer and label standardization

?

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
loss_before = criterion(outputs, labels)

print("Loss before:", loss_before.item())

optimizer.zero_grad()

loss_before.backward()

optimizer.step()

with torch.no_grad():
    outputs_after = model(images)
    loss_after = criterion(outputs_after, labels)

print("Loss after:", loss_after.item())

Loss before: 30322.560546875
Loss after: 30232.3515625


In [ ]:
images, labels = next(iter(loader))

with torch.no_grad():
    outputs = model(images)

print("========== INPUT STATISTICS ==========")

for i in range(images.shape[1]):
    channel = images[:, i, :, :]

    print(f"\nChannel {i+1}:")
    print("Min:", channel.min().item())
    print("Max:", channel.max().item())
    print("Mean:", channel.mean().item())

print("\n========== LABEL STATISTICS ==========")
print("Min:", labels.min().item())
print("Max:", labels.max().item())
print("Mean:", labels.mean().item())

print("\n========== MODEL OUTPUT STATISTICS ==========")
print("Min:", outputs.min().item())
print("Max:", outputs.max().item())
print("Mean:", outputs.mean().item())

========== INPUT STATISTICS ==========

Channel 1:
Min: 0.0
Max: 0.9552000164985657
Mean: 0.10840662568807602

Channel 2:
Min: 0.0
Max: 0.7960000038146973
Mean: 0.1318841129541397

Channel 3:
Min: 0.0
Max: 0.7063999772071838
Mean: 0.12177646905183792

Channel 4:
Min: 0.011800000444054604
Max: 0.6367999911308289
Mean: 0.3047903776168823

Channel 5:
Min: -1.0
Max: 0.31963470578193665
Mean: -0.427031934261322

Channel 6:
Min: nan
Max: nan
Mean: nan

========== LABEL STATISTICS ==========
Min: 5.0
Max: 293.6000061035156
Mean: 156.1649932861328

========== MODEL OUTPUT STATISTICS ==========
Min: nan
Max: nan
Mean: nan


In [ ]:
# Calculate mean and standard deviation from all dataset labels
all_labels = torch.tensor([dataset[i][1].item() for i in range(len(dataset))])

label_mean = all_labels.mean()
label_std = all_labels.std()

print("Label mean:", label_mean.item())
print("Label standard deviation:", label_std.item())


# Get one batch
images, labels = next(iter(loader))

# Standardize labels
labels = (labels - label_mean) / label_std

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Loss before training
outputs = model(images)
loss_before = criterion(outputs, labels.unsqueeze(1))

print("\nLoss before:", loss_before.item())

# Exactly one training step
optimizer.zero_grad()
loss_before.backward()
optimizer.step()

# Loss after training
with torch.no_grad():
    outputs_after = model(images)
    loss_after = criterion(outputs_after, labels.unsqueeze(1))

print("Loss after:", loss_after.item())

# Convert a standardized prediction back to NTU
predicted_ntu = outputs_after * label_std + label_mean

print("Predicted NTU:", predicted_ntu[0].item())

Label mean: 181.73428344726562
Label standard deviation: 96.80386352539062

Loss before: 1.2533605098724365
Loss after: 0.9984958171844482
Predicted NTU: 181.69427490234375


In [ ]:
from torch.utils.data import random_split

# 1. Calculate split sizes
total_size = len(dataset)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

# 2. Split dataset
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

# 3. Print split sizes
print("Total samples:", total_size)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

# 4. Get NTU labels from training dataset only
train_labels = torch.tensor([
    train_dataset[i][1].item()
    for i in range(len(train_dataset))
])

# 5. Calculate training label statistics
label_mean = train_labels.mean()
label_std = train_labels.std()

print("\nTraining label mean:", label_mean.item())
print("Training label std:", label_std.item())

Total samples: 2886
Training samples: 2308
Validation samples: 578

Training label mean: 179.50538635253906
Training label std: 97.1866226196289
